In [ ]:
# NOTEBOOK 03 — CONTEXT DATA EDA
# Exploratorio inicial y limpieza de datos de contexto

"""
este notebook realiza el exploratorio inicial y la limpieza de los datasets de contexto:
hospitalizaciones, mortalidad, población y estaciones de calidad del aire.

aquí dejamos cada dataset preparado, limpio y reducido a las columnas esenciales para
el análisis posterior
no se realizan merges ni integración con los contaminantes en este notebook

el análisis integrado (contaminantes + estaciones + contexto) se desarrollará en el Notebook 04.
"""

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)


In [2]:
# cargo datasets

df_hospi = pd.read_csv("../data/hospitalizations_spain.csv") # solo hospitalizaciones por problemas respiratorios
df_mortality = pd.read_csv("../data/mortality_respiratory_spain.csv")
df_population = pd.read_csv("../data/population_spain.csv")
df_stations = pd.read_csv("../data/stations_metadata_spain.csv")

print("Datasets cargados correctamente.")

Datasets cargados correctamente.


In [3]:
'''
los datasets de hospitalizaciones, mortalidad y población provienen de fuentes oficiales
(Eurostat/INE) y están altamente agregados. Por ello no presentan problemas típicos de
datos crudos (missings, duplicados, tipos incorrectos, outliers)
'''

'\nlos datasets de hospitalizaciones, mortalidad y población provienen de fuentes oficiales\n(Eurostat/INE) y están altamente agregados. Por ello no presentan problemas típicos de\ndatos crudos (missings, duplicados, tipos incorrectos, outliers)\n'

In [3]:
# hago una función para hacer un exploratorio inicial con los datos

def exploratorio(df, name):
    print("="*80)
    print(f"EXPLORATORIO INICIAL — {name}")
    print("="*80)

    # shape
    print("\nShape (filas, columnas):")
    print(df.shape)
    # info
    print("\nInfo:")
    print(df.info())
    # primeras filas
    print("\nPrimeras filas:")
    print(df.head())
    # tipos de datos
    print("\nTipos de datos:")
    print(df.dtypes)
    # missings
    print("\nPorcentaje de missings por columna:")
    print((df.isna().mean() * 100).sort_values(ascending=False))
    # duplicados
    print("\nNúmero de duplicados:")
    print(df.duplicated().sum())
    # valores únicos
    print("\nNúmero de valores únicos por columna:")
    print(df.nunique())


In [4]:
exploratorio(df_hospi, "Hospitalizaciones")

EXPLORATORIO INICIAL — Hospitalizaciones

Shape (filas, columnas):
(22, 25)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 25 columns):
 #   Column                                                                                     Non-Null Count  Dtype  
---  ------                                                                                     --------------  -----  
 0   STRUCTURE                                                                                  22 non-null     str    
 1   STRUCTURE_ID                                                                               22 non-null     str    
 2   STRUCTURE_NAME                                                                             22 non-null     str    
 3   freq                                                                                       22 non-null     str    
 4   Time frequency                                                                             22 non-nu

In [5]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE HOSPITALIZACIONES

Shape (22, 25): El dataset contiene 22 filas y 25 columnas. Cada fila representa un año, no una provincia ni un hospital
    No es un dataset territorial ni diario; es puramente temporal y nacional

Type: La mayoría de columnas son tipo string porque corresponden a metadatos
    administrativos (estructura, edad, sexo, unidad, clasificación ICD-10, etc.)
    Las únicas columnas numéricas reales son TIME_PERIOD (año) y OBS_VALUE (número de hospitalizaciones)

Primeras filas: Se observa que todas las columnas de metadatos contienen siempre el mismo valor: edad = TOTAL; sexo = T (Total);
    unidad = Number; icd10 = J (enfermedades respiratorias), indicador = in-patients (total number)
    El dataset ya viene filtrado: contiene únicamente hospitalizaciones por enfermedades respiratorias (ICD-10 J00-J99) para toda España
    Lo descargué así porque se me petaba el ordenador si descargaba todo.
    Hice la comprobación de si los datos de OBS_VALUE son coherentes con hospitalizaciones anuales (entre 400k y 470k) y sí lo son

Missings: Varias columnas están al 100% vacías (Time, Observation value, OBS_FLAG, CONF_STATUS…), no aportan información y se eliminarán

Duplicados: No hay duplicados, cada fila representa un año único

Valores únicos: La mayoría de columnas tienen un único valor (son metadatos constantes), para este EDA no aportan variabilidad analítica
    Las únicas columnas con variabilidad real son: TIME_PERIOD (22 años), OBS_VALUE (22 valores) y geo (aunque solo tiene "ES", es la clave territorial y peude ser imporante)
'''


'\nINTERPRETACIÓN DEL EXPLORATORIO DE HOSPITALIZACIONES\n\nShape (22, 25): El dataset contiene 22 filas y 25 columnas. Cada fila representa un año, no una provincia ni un hospital\n    No es un dataset territorial ni diario; es puramente temporal y nacional\n\nType: La mayoría de columnas son tipo string porque corresponden a metadatos\n    administrativos (estructura, edad, sexo, unidad, clasificación ICD-10, etc.)\n    Las únicas columnas numéricas reales son TIME_PERIOD (año) y OBS_VALUE (número de hospitalizaciones)\n\nPrimeras filas: Se observa que todas las columnas de metadatos contienen siempre el mismo valor: edad = TOTAL; sexo = T (Total);\n    unidad = Number; icd10 = J (enfermedades respiratorias), indicador = in-patients (total number)\n    El dataset ya viene filtrado: contiene únicamente hospitalizaciones por enfermedades respiratorias (ICD-10 J00-J99) para toda España\n    Lo descargué así porque se me petaba el ordenador si descargaba todo.\n    Hice la comprobación de

In [5]:
df_hospi.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'age', 'Age class', 'indic_he', 'Health indicator', 'unit',
       'Unit of measure', 'sex', 'Sex', 'icd10',
       'International Statistical Classification of Diseases and Related Health Problems (ICD-10)',
       'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'Time',
       'OBS_VALUE', 'Observation value', 'OBS_FLAG',
       'Observation status (Flag) V2 structure', 'CONF_STATUS',
       'Confidentiality status (flag)'],
      dtype='str')

In [6]:
df_hospi.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,age,Age class,indic_he,Health indicator,unit,Unit of measure,sex,Sex,icd10,International Statistical Classification of Diseases and Related Health Problems (ICD-10),geo,Geopolitical entity (reporting),TIME_PERIOD,Time,OBS_VALUE,Observation value,OBS_FLAG,Observation status (Flag) V2 structure,CONF_STATUS,Confidentiality status (flag)
0,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2000,NaN,432758,NaN,NaN,NaN,NaN,NaN
1,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2001,NaN,410312,NaN,NaN,NaN,NaN,NaN
2,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2002,NaN,438888,NaN,NaN,NaN,NaN,NaN
3,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2003,NaN,467244,NaN,NaN,NaN,NaN,NaN
4,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2004,NaN,449480,NaN,NaN,NaN,NaN,NaN


In [8]:
'''
antes de nada, concretar que la columna 'International Statistical Classification of Diseases and Related Health Problems (ICD-10)'
se refiere al tipo de ingreso y en todas las ocasiones es: 'Diseases of the respiratory system (J00-J99)'
J00-J99 el código que se refiere a las hospitalizaciones por problemas respiratorios, puede incluir, infecciones respiratorias,
asma, EPOC, neumonía, bronquitis u otras patologías respiratorias
'''

"\nantes de nada, concretar que la columna 'International Statistical Classification of Diseases and Related Health Problems (ICD-10)'\nse refiere al tipo de ingreso y en todas las ocasiones es: 'Diseases of the respiratory system (J00-J99)'\nJ00-J99 el código que se refiere a las hospitalizaciones por problemas respiratorios, puede incluir, infecciones respiratorias,\nasma, EPOC, neumonía, bronquitis u otras patologías respiratorias\n"

In [5]:
# veo que tengo columnas que tienen todo NaN, pero no estoy segura, 
# para eso veo qué columnas están al 100% a NaN

nan_cols = df_hospi.columns[df_hospi.isna().all()]
print("Columnas completamente NaN:")
print(nan_cols.tolist())

Columnas completamente NaN:
['Time', 'Observation value', 'OBS_FLAG', 'Observation status (Flag) V2 structure', 'CONF_STATUS', 'Confidentiality status (flag)']


In [6]:
# no se trata de datos importantes así que voy a eliminar esas columnas
# lo elimino del data inicial(df_hospi) porque quiero que quede limpio, si necesito el original má adelante, es vovler a ejecutar la celda del inicio

df_hospi = df_hospi.drop(columns=nan_cols) 

df_hospi.shape

(22, 19)

In [7]:
df_hospi.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'age', 'Age class', 'indic_he', 'Health indicator', 'unit',
       'Unit of measure', 'sex', 'Sex', 'icd10',
       'International Statistical Classification of Diseases and Related Health Problems (ICD-10)',
       'geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'OBS_VALUE'],
      dtype='str')

In [8]:
df_hospi.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,age,Age class,indic_he,Health indicator,unit,Unit of measure,sex,Sex,icd10,International Statistical Classification of Diseases and Related Health Problems (ICD-10),geo,Geopolitical entity (reporting),TIME_PERIOD,OBS_VALUE
0,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2000,432758
1,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2001,410312
2,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2002,438888
3,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2003,467244
4,dataflow,ESTAT:HLTH_CO_DISCH1(1.0),"Hospital discharges by diagnosis, in-patients,...",A,Annual,TOTAL,Total,INPAT,in-patients (total number),NR,Number,T,Total,J,Diseases of the respiratory system (J00-J99),ES,Spain,2004,449480


In [9]:
# ahora, siguiente paso

'''
en el exploratorio vemos que muchas columnas tienen solo 1 valor único, por lo que no aportan variabilidad, no contienen
información analítica, no sirven para agrupar, filtrar ni modelar y solo describen el dataset (metadatos de Eurostat),
por eso deben eliminarse
'''

# voy a identificar las  columnas que tienen un único valor

cols_one_value = [col for col in df_hospi.columns 
                  if df_hospi[col].nunique() == 1]

print(f'Columnas con un único valor: {cols_one_value}')

Columnas con un único valor: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency', 'age', 'Age class', 'indic_he', 'Health indicator', 'unit', 'Unit of measure', 'sex', 'Sex', 'icd10', 'International Statistical Classification of Diseases and Related Health Problems (ICD-10)', 'geo', 'Geopolitical entity (reporting)']


In [10]:
'''
columnas que deben eliminarse:
    metadatos administrativos de Eurostat: STRUCTURE, STRUCTURE_ID, STRUCTURE_NAME, freq, Time frequency
    metadatos de población/edad/sexo: age, Age class, sex, Sex
    metadatos del indicador: indic_he,Health indicator, unit, Unit of measure
    metadatos del diagnóstico: icd10, International Statistical Classification of Diseases and Related Health Problems (ICD-10)
    metadatos territoriales duplicados: Geopolitical entity (reporting)

son interesantes para contextualizar que se refieren a datos anuales, a todos los géneros y a todo tipo de edades de la población, etc.
pero a nivel análisis no suponen nada, al menos en este EDA
'''

# LA ÚNICA COLUMNA QUE NO VOY A ELIMINAR, aunque tenga un valor único, es geo.
# Solo contiene "ES", pero es la clave territorial y la necesitaré para merges u otros datas en los que también esté.

# allá vamos:

# filtro para no eliminar 'geo'
cols_to_drop = [col for col in cols_one_value if col != "geo"]

# elimino
df_hospi = df_hospi.drop(columns=cols_to_drop)


In [11]:
print("\nColumnas restantes después de la limpieza:")
print(df_hospi.columns.tolist())


Columnas restantes después de la limpieza:
['geo', 'TIME_PERIOD', 'OBS_VALUE']


In [12]:
# ahora voy a renombrar las columnas

df_hospi = df_hospi.rename(columns={
    "TIME_PERIOD": "year",
    "geo": "region",
    "OBS_VALUE": "hospitalizations"
})

print("Columnas renombradas:")
print(df_hospi.columns.tolist())


Columnas renombradas:
['region', 'year', 'hospitalizations']


In [13]:
df_hospi.head()

,region,year,hospitalizations
0,ES,2000,432758
1,ES,2001,410312
2,ES,2002,438888
3,ES,2003,467244
4,ES,2004,449480


In [14]:
df_hospi.dtypes

region                str
year                int64
hospitalizations    int64
dtype: object

In [15]:
# ahora filtro por año porque es el rango común con: contaminantes, mortalidad respiratoria y población

df_hospi = df_hospi[df_hospi["year"].between(2011, 2019)]

print("Shape después de filtrar por año:", df_hospi.shape)


Shape después de filtrar por año: (9, 3)


In [16]:
df_hospi

,region,year,hospitalizations
11,ES,2011,518276
12,ES,2012,524962
13,ES,2013,506929
14,ES,2014,531124
15,ES,2015,426914
16,ES,2016,586945
17,ES,2017,594741
18,ES,2018,635132
19,ES,2019,600088


In [23]:
# el dataset de hospitalizaciones por causas respiratorias ya está limpio

# ahora haremos lo mismo con en resto de datas

In [17]:
exploratorio(df_mortality, "Mortalidad")

EXPLORATORIO INICIAL — Mortalidad

Shape (filas, columnas):
(27, 25)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 25 columns):
 #   Column                                                                                     Non-Null Count  Dtype  
---  ------                                                                                     --------------  -----  
 0   STRUCTURE                                                                                  27 non-null     str    
 1   STRUCTURE_ID                                                                               27 non-null     str    
 2   STRUCTURE_NAME                                                                             27 non-null     str    
 3   freq                                                                                       27 non-null     str    
 4   Time frequency                                                                             27 non-null     

In [25]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE MORTALIDAD

Shape (27, 25): El dataset contiene 27 filas y 25 columnas. Cada fila representa un año.
    Es un dataset nacional, no territorial

Type: La mayoría de columnas son strings porque son metadatos administrativos (estructura, sexo, edad, unidad, ICD-10, residencia…)
    Las únicas columnas numéricas reales son TIME_PERIOD (año) y OBS_VALUE (número de muertes)

Primeras filas: Todas las columnas de metadatos contienen siempre el mismo valor: edad = TOTAL; sexo = T (Total); unidad = Number;
    icd10 = J (enfermedades respiratorias), resid = TOT_IN (todas las muertes registradas en el país)
    El dataset ya viene filtrado a mortalidad respiratoria (ICD-10 J00–J99) para toda España. Lo descargué así porque se me petaba el ordenador si descargaba todo.
    OBS_VALUE contiene valores coherentes con mortalidad respiratoria anual (entre 40k y 50k).

Missings: Varias columnas están al 100% vacías (Time, Observation value, OBS_FLAG, CONF_STATUS…), no aportan información y se eliminarán.

Duplicados: No hay duplicados, cada fila representa un año único.

Valores únicos: La mayoría de columnas tienen un único valor (son metadatos constantes), para este EDA no aportan variabilidad analítica
    Las únicas columnas con variabilidad real son: TIME_PERIOD (14 años); OBS_VALUE (14 valores); resid (2 valores, pero no aporta variabilidad útil)
    y geo (solo “ES”, pero es clave territorial)

'''


'\nINTERPRETACIÓN DEL EXPLORATORIO DE MORTALIDAD\n\nShape (27, 25): El dataset contiene 27 filas y 25 columnas. Cada fila representa un año.\n    Es un dataset nacional, no territorial\n\nType: La mayoría de columnas son strings porque son metadatos administrativos (estructura, sexo, edad, unidad, ICD-10, residencia…)\n    Las únicas columnas numéricas reales son TIME_PERIOD (año) y OBS_VALUE (número de muertes)\n\nPrimeras filas: Todas las columnas de metadatos contienen siempre el mismo valor: edad = TOTAL; sexo = T (Total); unidad = Number;\n    icd10 = J (enfermedades respiratorias), resid = TOT_IN (todas las muertes registradas en el país)\n    El dataset ya viene filtrado a mortalidad respiratoria (ICD-10 J00–J99) para toda España. Lo descargué así porque se me petaba el ordenador si descargaba todo.\n    OBS_VALUE contiene valores coherentes con mortalidad respiratoria anual (entre 40k y 50k).\n\nMissings: Varias columnas están al 100% vacías (Time, Observation value, OBS_FLAG, 

In [18]:
df_mortality.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'unit', 'Unit of measure', 'sex', 'Sex', 'age', 'Age class', 'icd10',
       'International Statistical Classification of Diseases and Related Health Problems (ICD-10)',
       'resid', 'Place of residence', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD', 'Time', 'OBS_VALUE', 'Observation value', 'OBS_FLAG',
       'Observation status (Flag) V2 structure', 'CONF_STATUS',
       'Confidentiality status (flag)'],
      dtype='str')

In [19]:
df_mortality.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,unit,Unit of measure,sex,Sex,age,Age class,icd10,International Statistical Classification of Diseases and Related Health Problems (ICD-10),resid,Place of residence,geo,Geopolitical entity (reporting),TIME_PERIOD,Time,OBS_VALUE,Observation value,OBS_FLAG,Observation status (Flag) V2 structure,CONF_STATUS,Confidentiality status (flag)
0,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2011,NaN,42243,NaN,NaN,NaN,NaN,NaN
1,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2012,NaN,47336,NaN,NaN,NaN,NaN,NaN
2,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2013,NaN,42565,NaN,NaN,NaN,NaN,NaN
3,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2014,NaN,43840,NaN,NaN,NaN,NaN,NaN
4,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2015,NaN,51848,NaN,NaN,NaN,NaN,NaN


In [20]:
# eliminar columnas con todo NaN
nan_cols_2 = df_mortality.columns[df_mortality.isna().all()]
print("Columnas 100% NaN:", nan_cols_2.tolist())

Columnas 100% NaN: ['Time', 'Observation value', 'OBS_FLAG', 'Observation status (Flag) V2 structure', 'CONF_STATUS', 'Confidentiality status (flag)']


In [21]:
df_mortality = df_mortality.drop(columns=nan_cols_2)

df_mortality.shape

(27, 19)

In [22]:
df_mortality.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency',
       'unit', 'Unit of measure', 'sex', 'Sex', 'age', 'Age class', 'icd10',
       'International Statistical Classification of Diseases and Related Health Problems (ICD-10)',
       'resid', 'Place of residence', 'geo', 'Geopolitical entity (reporting)',
       'TIME_PERIOD', 'OBS_VALUE'],
      dtype='str')

In [23]:
df_mortality.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,unit,Unit of measure,sex,Sex,age,Age class,icd10,International Statistical Classification of Diseases and Related Health Problems (ICD-10),resid,Place of residence,geo,Geopolitical entity (reporting),TIME_PERIOD,OBS_VALUE
0,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2011,42243
1,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2012,47336
2,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2013,42565
3,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2014,43840
4,dataflow,ESTAT:HLTH_CD_ARO(1.0),Causes of death - deaths by country of residen...,A,Annual,NR,Number,T,Total,TOTAL,Total,J,Diseases of the respiratory system (J00-J99),TOT_IN,All deaths reported in the country,ES,Spain,2015,51848


In [24]:
# identificar columnas con un único valor
cols_one_value_2 = [col for col in df_mortality.columns if df_mortality[col].nunique() == 1]
print("Columnas con un único valor:", cols_one_value_2)

Columnas con un único valor: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency', 'unit', 'Unit of measure', 'sex', 'Sex', 'age', 'Age class', 'icd10', 'International Statistical Classification of Diseases and Related Health Problems (ICD-10)', 'geo', 'Geopolitical entity (reporting)']


In [25]:
# eliminar columnas constantes excepto 'geo'
cols_to_drop_2 = [col for col in cols_one_value_2 if col != "geo"]

df_mortality = df_mortality.drop(columns=cols_to_drop_2)

In [26]:
print("\nColumnas restantes después de la limpieza:")
print(df_mortality.columns.tolist())


Columnas restantes después de la limpieza:
['resid', 'Place of residence', 'geo', 'TIME_PERIOD', 'OBS_VALUE']


In [27]:
# renombrar columnas
df_mortality = df_mortality.rename(columns={
    "TIME_PERIOD": "year",
    "geo": "region",
    "OBS_VALUE": "deaths"
})

print("Columnas tras renombrar:", df_mortality.columns.tolist())

Columnas tras renombrar: ['resid', 'Place of residence', 'region', 'year', 'deaths']


In [28]:
df_mortality.dtypes

resid                   str
Place of residence      str
region                  str
year                  int64
deaths                int64
dtype: object

In [29]:
df_mortality.head()

,resid,Place of residence,region,year,deaths
0,TOT_IN,All deaths reported in the country,ES,2011,42243
1,TOT_IN,All deaths reported in the country,ES,2012,47336
2,TOT_IN,All deaths reported in the country,ES,2013,42565
3,TOT_IN,All deaths reported in the country,ES,2014,43840
4,TOT_IN,All deaths reported in the country,ES,2015,51848


In [30]:
# filtrar rango temporal 2011–2019
df_mortality = df_mortality[df_mortality["year"].between(2011, 2019)]

print("Shape después de filtrar por año:", df_mortality.shape)

Shape después de filtrar por año: (18, 5)


In [31]:
df_mortality

,resid,Place of residence,region,year,deaths
0,TOT_IN,All deaths reported in the country,ES,2011,42243
1,TOT_IN,All deaths reported in the country,ES,2012,47336
2,TOT_IN,All deaths reported in the country,ES,2013,42565
3,TOT_IN,All deaths reported in the country,ES,2014,43840
4,TOT_IN,All deaths reported in the country,ES,2015,51848
5,TOT_IN,All deaths reported in the country,ES,2016,46812
6,TOT_IN,All deaths reported in the country,ES,2017,51615
7,TOT_IN,All deaths reported in the country,ES,2018,53687
8,TOT_IN,All deaths reported in the country,ES,2019,47681
14,TOT_RESID,All deaths of residents in or outside their ho...,ES,2011,42125


In [32]:
# una vez está filtrado por años, vamos a terminar de arreglar el df porque, si observamos la tabla del output que sale
# de poner df_mortality, vemos que hay dos datos para cada año y puede ser confuso

'''
la columna resid es un código de clasificación que indica cómo se contabilizan las muertes según el lugar de residencia o el
lugar donde ocurrió el fallecimiento. en el dataset aparecen dos códigos:

    TOT_IN: todas las muertes ocurridas en España (útil para hospitales, emergencias
    TOT_RESID: muertes de residentes españoles, independientemente de dónde murieron (útil para epidemiología poblacional)


la columna Place of residence es la descripción en texto del código resid (no aporta información nueva, solo explica el código):
    para TOT_IN es “All deaths reported in the country”
    para TOT_RESID es “All deaths of residents in or outside their home country”


TOT_IN y TOT_RESID son válidas ambas, pero no se deben mezclar

para un análisis de salud pública basado en población, lo correcto es usar: TOT_RESID porque hospitalizaciones -> son residentes;
tasas por 100k habitantes -> requieren residentes; mortalidad respiratoria -> debe ser de residentes

'''

# filtro solo TOT_RESID:

df_mortality = df_mortality[df_mortality["resid"] == "TOT_RESID"]

# y ahora voy a eliminar las columnas resid y Place of residence porque ya no aportan nada:
    
df_mortality = df_mortality.drop(columns=["resid", "Place of residence"])


In [33]:
df_mortality.shape

(9, 3)

In [34]:
df_mortality

,region,year,deaths
14,ES,2011,42125
15,ES,2012,47237
16,ES,2013,42444
17,ES,2014,43699
18,ES,2015,51711
19,ES,2016,46649
20,ES,2017,51413
21,ES,2018,53476
22,ES,2019,47470


In [35]:
df_mortality.dtypes

region      str
year      int64
deaths    int64
dtype: object

In [ ]:
# mortalidad ya está

# pasamos a población

In [36]:
exploratorio(df_population, "Población")

EXPLORATORIO INICIAL — Población

Shape (filas, columnas):
(6, 31)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 31 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Series Name    3 non-null      str    
 1   Series Code    1 non-null      str    
 2   Country Name   1 non-null      str    
 3   Country Code   1 non-null      str    
 4   1990 [YR1990]  1 non-null      float64
 5   2000 [YR2000]  1 non-null      float64
 6   2001 [YR2001]  1 non-null      float64
 7   2002 [YR2002]  1 non-null      float64
 8   2003 [YR2003]  1 non-null      float64
 9   2004 [YR2004]  1 non-null      float64
 10  2005 [YR2005]  1 non-null      float64
 11  2006 [YR2006]  1 non-null      float64
 12  2007 [YR2007]  1 non-null      float64
 13  2008 [YR2008]  1 non-null      float64
 14  2009 [YR2009]  1 non-null      float64
 15  2010 [YR2010]  1 non-null      float64
 16  2011 [YR2011]  1 non-null      float64


In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE POBLACIÓN (World Bank)

Shape (6, 31): El dataset contiene 6 filas y 31 columnas. Esto indica que no es un dataset “tidy”, 
sino un fichero típico del Banco Mundial con una única fila útil y varias filas de notas o metadatos.

Estructura: La fila 0 contiene los datos reales de población de España desde 1990 hasta 2025. 
Las filas restantes están vacías o contienen mensajes administrativos del proveedor de datos. 
Las columnas de años están en formato wide (“2011 [YR2011]”), lo que confirma que la tabla 
no está lista para análisis directo y requiere transformación.

Primeras filas: Solo la fila 0 tiene valores numéricos coherentes con la población total de España 
(entre 38 y 48 millones según el año). El resto de filas contienen NaNs o textos informativos 
que no aportan valor analítico. Las columnas de metadatos (Series Name, Country Name, etc.) 
son constantes y no añaden variabilidad.

Missings: La mayoría de columnas presentan un 83% de valores nulos porque solo una fila contiene datos. 
Esto no implica ausencia de información, sino que el dataset incluye filas vacías que deben eliminarse. 
Los NaNs no afectan a la calidad de los datos reales, pero sí requieren limpieza previa.

Duplicados: Se detectan 2 duplicados, pero no son duplicados reales de datos, sino filas vacías repetidas. 
Se eliminarán sin impacto en el análisis.

Valores únicos: Todas las columnas de años tienen un único valor (la población de ese año). 
Las columnas de metadatos también tienen un único valor, lo que confirma que no aportan variabilidad 
y deben eliminarse para dejar solo la serie temporal.

Conclusión: El dataset contiene información válida y coherente de población total de España, 
pero está en formato wide y mezclado con metadatos. Para su uso en el EDA, es necesario:
    - quedarnos únicamente con la fila 0,
    - eliminar filas vacías y columnas irrelevantes,
    - transformar los años a formato tidy (year, population).

Una vez limpio, el dataset será adecuado para calcular tasas por 100.000 habitantes y 
relacionar población con hospitalizaciones y mortalidad en el análisis de salud (H1)

'''

'\nINTERPRETACIÓN DEL EXPLORATORIO DE POBLACIÓN\n\nShape (22, 25): El dataset contiene 22 filas y 25 columnas. Igual que hospitalizaciones, cada fila representa un año\n    Es un dataset nacional, no territorial\n\nType: La mayoría de columnas son strings porque son metadatos administrativos (estructura, edad, sexo, unidad, ICD-10, etc.)\n    Las únicas columnas numéricas reales son TIME_PERIOD (año) y OBS_VALUE (población total)\n\nPrimeras filas: Todas las columnas de metadatos tienen siempre el mismo valor: edad = TOTAL; sexo = T (Total); unidad = Number;\n    e icd10 = J (aunque aquí no tiene sentido, es un residuo del formato Eurostat)\n    El dataset es de la población total nacional\n    Después de comprobarlo, OBS_VALUE contiene valores coherentes con población total de España (entre 40 y 47 millones según el año)\n\nMissings: Varias columnas están al 100% vacías (Time, Observation value, OBS_FLAG, CONF_STATUS…), no aportan información y se eliminarán\n\nDuplicados: No hay dupl

In [37]:
df_population.columns

Index(['Series Name', 'Series Code', 'Country Name', 'Country Code',
       '1990 [YR1990]', '2000 [YR2000]', '2001 [YR2001]', '2002 [YR2002]',
       '2003 [YR2003]', '2004 [YR2004]', '2005 [YR2005]', '2006 [YR2006]',
       '2007 [YR2007]', '2008 [YR2008]', '2009 [YR2009]', '2010 [YR2010]',
       '2011 [YR2011]', '2012 [YR2012]', '2013 [YR2013]', '2014 [YR2014]',
       '2015 [YR2015]', '2016 [YR2016]', '2017 [YR2017]', '2018 [YR2018]',
       '2019 [YR2019]', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]',
       '2023 [YR2023]', '2024 [YR2024]', '2025 [YR2025]'],
      dtype='str')

In [38]:
df_population.head()

,Series Name,Series Code,Country Name,Country Code,1990 [YR1990],2000 [YR2000],2001 [YR2001],2002 [YR2002],2003 [YR2003],2004 [YR2004],2005 [YR2005],2006 [YR2006],2007 [YR2007],2008 [YR2008],2009 [YR2009],2010 [YR2010],2011 [YR2011],2012 [YR2012],2013 [YR2013],2014 [YR2014],2015 [YR2015],2016 [YR2016],2017 [YR2017],2018 [YR2018],2019 [YR2019],2020 [YR2020],2021 [YR2021],2022 [YR2022],2023 [YR2023],2024 [YR2024],2025 [YR2025]
0,"Population, total",SP.POP.TOTL,Spain,ESP,38867322.0,40567864.0,40850412.0,41431558.0,42187645.0,42921895.0,43653155.0,44397319.0,45226803.0,45954106.0,46362946.0,46576897.0,46742697.0,46773055.0,46604197.0,46460733.0,46422303.0,46458139.0,46571232.0,46782011.0,47118501.0,47359424.0,47443821.0,47786102.0,48352528.0,48848840.0,..
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Data from database: World Development Indicators,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
# me quedo únicamente con la fila que contiene datos reales
df_population = df_population.iloc[[0]].copy()

In [40]:
# elimino columnas de metadatos que no aportan información (Series Name, Series Code, Country Name, Country Code)
cols_to_drop = ['Series Name', 'Series Code', 'Country Name', 'Country Code']
df_population = df_population.drop(columns=cols_to_drop)

In [41]:
df_population.columns

Index(['1990 [YR1990]', '2000 [YR2000]', '2001 [YR2001]', '2002 [YR2002]',
       '2003 [YR2003]', '2004 [YR2004]', '2005 [YR2005]', '2006 [YR2006]',
       '2007 [YR2007]', '2008 [YR2008]', '2009 [YR2009]', '2010 [YR2010]',
       '2011 [YR2011]', '2012 [YR2012]', '2013 [YR2013]', '2014 [YR2014]',
       '2015 [YR2015]', '2016 [YR2016]', '2017 [YR2017]', '2018 [YR2018]',
       '2019 [YR2019]', '2020 [YR2020]', '2021 [YR2021]', '2022 [YR2022]',
       '2023 [YR2023]', '2024 [YR2024]', '2025 [YR2025]'],
      dtype='str')

In [42]:
# extraigo el año de los nombres de columna
df_population.columns = df_population.columns.str.extract(r'(\d{4})')[0]


In [43]:
# elimino columnas que no son años (quedan como NaN tras la extracción)
df_population = df_population.dropna(axis=1, how='all')

In [44]:
# ahora convierto la fila única en formato tidy: columnas = años y filas = año, población
df_population = df_population.melt(
    var_name='year',
    value_name='population')

In [45]:
df_population

,year,population
0,1990,38867322.0
1,2000,40567864.0
2,2001,40850412.0
3,2002,41431558.0
4,2003,42187645.0
5,2004,42921895.0
6,2005,43653155.0
7,2006,44397319.0
8,2007,45226803.0
9,2008,45954106.0


In [46]:
# eliminar filas donde population sea '..', que es la de 2025
df_population = df_population[df_population['population'] != '..']

In [47]:
# conversión de tipos
# año -> int
# población -> int
df_population['year'] = df_population['year'].astype(int)
df_population['population'] = df_population['population'].astype(int)

In [48]:
df_population.dtypes

year          int64
population    int64
dtype: object

In [49]:
# hora de filtrar los años necesarios (2011–2019)
df_population = df_population[df_population['year'].between(2011, 2019)]

In [50]:
df_population

,year,population
12,2011,46742697
13,2012,46773055
14,2013,46604197
15,2014,46460733
16,2015,46422303
17,2016,46458139
18,2017,46571232
19,2018,46782011
20,2019,47118501


In [51]:
# añado la columna region "ES"
df_population['region'] = 'ES'


# ahora voy a reordenar las columnas al formato final que quiero
df_population = df_population[['region', 'year', 'population']]


df_population

,region,year,population
12,ES,2011,46742697
13,ES,2012,46773055
14,ES,2013,46604197
15,ES,2014,46460733
16,ES,2015,46422303
17,ES,2016,46458139
18,ES,2017,46571232
19,ES,2018,46782011
20,ES,2019,47118501


In [52]:
df_population.dtypes

region          str
year          int64
population    int64
dtype: object

In [ ]:
# dataset de population listo

In [53]:
exploratorio(df_stations, "Estaciones")

EXPLORATORIO INICIAL — Estaciones

Shape (filas, columnas):
(8564, 70)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 8564 entries, 0 to 8563
Data columns (total 70 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Country                       8564 non-null   str    
 1   B-G Namespace                 8564 non-null   str    
 2   Year                          8564 non-null   int64  
 3   Air Quality Network           8564 non-null   str    
 4   Air Quality Network Name      8564 non-null   str    
 5   Timezone                      8564 non-null   str    
 6   Air Quality Station EoI Code  8564 non-null   str    
 7   Air Quality Station Nat Code  8564 non-null   int64  
 8   Air Quality Station Name      8564 non-null   str    
 9   Sampling Point Id             8564 non-null   str    
 10  Air Pollutant                 8564 non-null   str    
 11  Longitude                     8564 non-null   float64


In [ ]:
'''
INTERPRETACIÓN DEL EXPLORATORIO DE ESTACIONES

Shape (8564, 70): Dataset muy grande: 8.564 registros y 70 columnas. Cada fila es una combinación estación-contaminante-año
    Representa estaciones de calidad del aire, no valores de contaminación

Type: Mezcla de columnas numéricas (coordenadas, alturas, distancias, emisiones) y columnas categóricas (tipo de estación,
    municipio, método de medición…). Es un dataset heterogéneo y técnico

Primeras filas: Se observan múltiples contaminantes por estación (PM10, NO2, O3, metales…)
    Las coordenadas y características físicas de la estación están completas
    Muchas columnas describen: tipo de estación (tráfico, urbana, suburbana), método de medición, distancias a carreteras o
    edificios, emisiones locales
    Es un dataset descriptivo de infraestructura, no de mediciones diarias

Missings: Muchas columnas están al 100% vacías (Process Activity End, Other Analytical Technique…)
    Otras tienen valores -999, que representan “dato no disponible”
    Requiere limpieza selectiva

Duplicados: No hay duplicados exactos, pero sí múltiples filas por estación (una por contaminante)

Valores únicos: Algunas columnas tienen 1 único valor (Country, Namespace, Cadence…), otras tienen alta variabilidad (Air Pollutant, Station Name, Municipality)

'''


In [54]:
df_stations.columns


Index(['Country', 'B-G Namespace', 'Year', 'Air Quality Network',
       'Air Quality Network Name', 'Timezone', 'Air Quality Station EoI Code',
       'Air Quality Station Nat Code', 'Air Quality Station Name',
       'Sampling Point Id', 'Air Pollutant', 'Longitude', 'Latitude',
       'Altitude', 'Altitude Unit', 'Air Quality Station Area',
       'Air Quality Station Type', 'Operational Activity Begin',
       'Operational Activity End', 'Sample Id', 'Inlet Height',
       'Inlet Height Unit', 'Building Distance', 'Building Distance Unit',
       'Kerb Distance', 'Kerb Distance Unit', 'Distance Source',
       'Distance Source Unit', 'Main Emission Sources', 'Heating Emissions',
       'Heating Emissions Unit', 'Mobile', 'Traffic Emissions',
       'Traffic Emissions Unit', 'Industrial Emissions',
       'Industrial Emissions Unit', 'Municipality', 'Dispersion Local',
       'Dispersion Regional', 'Distance Junction', 'Distance Junction Unit',
       'Heavy Duty Fraction', 'Height 

In [55]:
df_stations.head()

,Country,B-G Namespace,Year,Air Quality Network,Air Quality Network Name,Timezone,Air Quality Station EoI Code,Air Quality Station Nat Code,Air Quality Station Name,Sampling Point Id,Air Pollutant,Longitude,Latitude,Altitude,Altitude Unit,Air Quality Station Area,Air Quality Station Type,Operational Activity Begin,Operational Activity End,Sample Id,Inlet Height,Inlet Height Unit,Building Distance,Building Distance Unit,Kerb Distance,Kerb Distance Unit,Distance Source,Distance Source Unit,Main Emission Sources,Heating Emissions,Heating Emissions Unit,Mobile,Traffic Emissions,Traffic Emissions Unit,Industrial Emissions,Industrial Emissions Unit,Municipality,Dispersion Local,Dispersion Regional,Distance Junction,Distance Junction Unit,Heavy Duty Fraction,Height Facades,Street Width,Traffic Speed,Traffic Volume,Process Id,Process Activity Begin,Process Activity End,Measurement Type,Measurement Method,Other Measurement Method,Measurement Equipment,Other Measurement Equipment,Sampling Method,Other Sampling Method,Analytical Technique,Other Analytical Technique,Equivalence Demonstrated,Demonstration Report,Detection Limit,Detection Limit Unit,Documentation,QA Report,Duration,Duration Unit,Cadence,Cadence Unit,Source Data URL,Imported
0,Spain,ES.BDCA.AQD,2025,NET_ES021A,Ayto Zaragoza,UTC,ES1044A,50297026,EL PICARRAL,SP_50297026_10_M,PM10,-0.87111,41.67028,195.0,m,suburban,traffic,01/01/2017 00:00:00,NaN,SAM_50297026_10_M,3.9,m,25.0,m,25.0,m,-999.0,m,Transport,-999.0,t.km-2.year-1,0,-999.0,t.km-1.year-1,-999.0,t.year-1,ZARAGOZA,detached,NaN,6.0,NaN,NaN,15.0,33.0,50,20620,SPP_50297026_10_M.1,01/01/2017 00:00:00,NaN,active,NaN,NaN,NaN,NaN,other,NaN,gravi,NaN,ref,NaN,10000.0,ug.m-3,15%,http://missing.com,1,day,1,day,http://cdr.eionet.europa.eu/es/eu/aqd/d/envaug...,17/12/2025 02:01:05
1,Spain,ES.BDCA.AQD,2025,NET_ES021A,Ayto Zaragoza,UTC,ES1044A,50297026,EL PICARRAL,SP_50297026_12_8,NOX as NO2,-0.87111,41.67028,195.0,m,suburban,traffic,01/01/2007 00:00:00,NaN,SAM_50297026_12_8,3.5,m,29.0,m,10.0,m,-999.0,m,Transport,-999.0,t.km-2.year-1,0,-999.0,t.km-1.year-1,-999.0,t.year-1,ZARAGOZA,detached,NaN,6.0,NaN,NaN,15.0,33.0,50,20620,SPP_50297026_12_8.1,01/01/2007 00:00:00,NaN,automatic,chemi,NaN,thermo42c,NaN,NaN,NaN,NaN,NaN,ref,NaN,0.4,ug.m-3,NaN,NaN,1,hour,1,hour,http://cdr.eionet.europa.eu/es/eu/aqd/d/envaug...,17/12/2025 02:01:05
2,Spain,ES.BDCA.AQD,2025,NET_ES021A,Ayto Zaragoza,UTC,ES1044A,50297026,EL PICARRAL,SP_50297026_14_6,O3,-0.87111,41.67028,195.0,m,suburban,traffic,01/03/1995 00:00:00,NaN,SAM_50297026_14_6,3.5,m,29.0,m,10.0,m,-999.0,m,Transport,-999.0,t.km-2.year-1,0,-999.0,t.km-1.year-1,-999.0,t.year-1,ZARAGOZA,detached,NaN,6.0,NaN,NaN,15.0,33.0,50,20620,SPP_50297026_14_6.1,01/03/1995 00:00:00,NaN,automatic,UV-P,NaN,thermo49i,NaN,NaN,NaN,NaN,NaN,ref,NaN,1.0,ug.m-3,"PARA 200PPB +/- 6,6",NaN,1,hour,1,hour,http://cdr.eionet.europa.eu/es/eu/aqd/d/envaug...,17/12/2025 02:01:05
3,Spain,ES.BDCA.AQD,2025,NET_ES021A,Ayto Zaragoza,UTC,ES1044A,50297026,EL PICARRAL,SP_50297026_17_M,As in PM10,-0.87111,41.67028,195.0,m,suburban,traffic,01/01/2017 00:00:00,NaN,SAM_50297026_17_M,4.0,m,7.6,m,5.6,m,-999.0,m,Transport,-999.0,t.km-2.year-1,0,-999.0,t.km-1.year-1,-999.0,t.year-1,ZARAGOZA,detached,NaN,6.0,NaN,NaN,15.0,33.0,50,20620,SPP_50297026_17_M.1,01/01/2017 00:00:00,NaN,active,NaN,NaN,NaN,NaN,HVSauto30,NaN,ICP-MS,NaN,ref,NaN,50.0,ng.m-3,20%,http://missing.com,1,day,1,day,http://cdr.eionet.europa.eu/es/eu/aqd/d/envaug...,17/12/2025 02:01:05
4,Spain,ES.BDCA.AQD,2025,NET_ES021A,Ayto Zaragoza,UTC,ES1044A,50297026,EL PICARRAL,SP_50297026_19_M,Pb in PM10,-0.87111,41.67028,195.0,m,suburban,traffic,01/01/2017 00:00:00,NaN,SAM_50297026_19_M,4.0,m,7.6,m,5.6,m,-999.0,m,Transport,-999.0,t.km-2.year-1,0,-999.0,t.km-1.year-1,-999.0,t.year-1,ZARAGOZA,detached,NaN,6.0,NaN,NaN,15.0,33.0,50,20620,SPP_50297026_19_M.1,01/01/2017 00:00:00,NaN,active,NaN,NaN,NaN,NaN,HVSauto30,NaN,ICP-MS,NaN,ref,NaN,50.0,ug.m-3,15%,http://missing.com,1,day,1,day,http://cdr.eionet.europa.

In [56]:
# he visto que hay columnas con todo NaN, así que voy a mirar

nan_cols_stat = df_stations.columns[df_stations.isna().all()]
print("Columnas completamente NaN:")
print(nan_cols_stat.tolist())


Columnas completamente NaN:
['Operational Activity End', 'Distance Junction Unit', 'Process Activity End', 'Other Measurement Method', 'Other Measurement Equipment', 'Other Sampling Method', 'Other Analytical Technique']


In [57]:
# vamos a eliminarlas porque no nos sirven de nada

df_stations = df_stations.drop(columns=nan_cols_stat)

print("Shape tras eliminar columnas vacías:", df_stations.shape)

Shape tras eliminar columnas vacías: (8564, 63)


In [58]:
# hora voyy a buscar columnas con un mismo valor

cols_one_value_stat = [col for col in df_stations.columns if df_stations[col].nunique() == 1]
print("Columnas con un único valor:", cols_one_value_stat)

Columnas con un único valor: ['Country', 'B-G Namespace', 'Year', 'Timezone', 'Altitude Unit', 'Inlet Height Unit', 'Building Distance Unit', 'Kerb Distance Unit', 'Distance Source Unit', 'Heating Emissions', 'Heating Emissions Unit', 'Mobile', 'Traffic Emissions', 'Traffic Emissions Unit', 'Industrial Emissions', 'Industrial Emissions Unit', 'Duration', 'Cadence', 'Source Data URL', 'Imported']


In [59]:
# no todas las columnas que tienen un valor único las voy a eliminar porque algunas son útiles para filtrar o documentar

In [60]:
# ahora eliminaré las columnas que no aportan nada

'''
algunas de las columnas con valor único, aunque técnicamente no están vacías, no aportan ninguna información útil para el análisis porque no permiten distinguir estaciones,
contaminantes ni características relevantes. además, sólo ocupan espacio y dificultan la lectura del dataframe

después de mirar columna a columna, he decidido eliminar:

Country: siempre es "Spain", todas las estaciones del dataset pertenecen a España, así que es redundante
    no aporta variabilidad ni sirve para filtrar nada.

B-G Namespace: es un identificador interno del sistema EEA/EIONET (es un metadato técnico del fichero, no del contenido)
    no tiene utilidad analítica ni aporta información sobre la estación

Cadence: siempre vale 1, indica la frecuencia de muestreo del inventario de estaciones, no de los datos de contaminación
    no se usa para filtrar ni para unir con otros datasets.

Source Data URL: es la URL del repositorio de donde se descargó el dataset (es idéntica en todas las filas)
    no aporta información sobre estaciones, contaminantes ni ubicación

Imported: es la fecha/hora en la que se importó el dataset al sistema de la EEA (es un metadato administrativo, no técnico)
    no aporta información útil para el análisis.
'''


cols_to_drop_stat = [
    "Country",
    "B-G Namespace",
    "Cadence",
    "Source Data URL",
    "Imported"
]

df_stations = df_stations.drop(columns=[col for col in cols_to_drop_stat if col in df_stations.columns])

print("Shape tras eliminar constantes irrelevantes:", df_stations.shape)

Shape tras eliminar constantes irrelevantes: (8564, 58)


In [61]:
df_stations.columns

Index(['Year', 'Air Quality Network', 'Air Quality Network Name', 'Timezone',
       'Air Quality Station EoI Code', 'Air Quality Station Nat Code',
       'Air Quality Station Name', 'Sampling Point Id', 'Air Pollutant',
       'Longitude', 'Latitude', 'Altitude', 'Altitude Unit',
       'Air Quality Station Area', 'Air Quality Station Type',
       'Operational Activity Begin', 'Sample Id', 'Inlet Height',
       'Inlet Height Unit', 'Building Distance', 'Building Distance Unit',
       'Kerb Distance', 'Kerb Distance Unit', 'Distance Source',
       'Distance Source Unit', 'Main Emission Sources', 'Heating Emissions',
       'Heating Emissions Unit', 'Mobile', 'Traffic Emissions',
       'Traffic Emissions Unit', 'Industrial Emissions',
       'Industrial Emissions Unit', 'Municipality', 'Dispersion Local',
       'Dispersion Regional', 'Distance Junction', 'Heavy Duty Fraction',
       'Height Facades', 'Street Width', 'Traffic Speed', 'Traffic Volume',
       'Process Id', 'Proces

In [62]:
# ahora eliminaré columnas técnicas que no aportan valor analítico

'''
no son columnas vacías, pero presentan metadatos extremadamente técnicos sobre unidades de medida,
formatos internos o descripciones redundantes

no aportan información útil para el análisis de contaminación ni para la selección de estaciones y solo
añaden ruido y complejidad al dataframe

tras observar bien el data, voy a eliminar:

Altitude Unit: siempre es "m", la unidad es redundante porque Altitude ya está en metros
    no aporta información adicional

Inlet Height Unit: siempre es "m", describe la unidad de la altura de entrada del muestreador
    no es relevante para el análisis de contaminación a nivel nacional

Building Distance Unit: siempre es "m", la unidad es redundante, la distancia ya está en metros
    no aporta variabilidad

Kerb Distance Unit: igual que la anterior: siempre "m"
    no aporta info útil

Distance Source Unit: siempre "m", es un metadato técnico del inventario, no del contaminante ni de la estación

Heating Emissions Unit: siempre "t.km-2.year-1", describe la unidad de emisiones de calefacción, pero no se usa en el análisis
    no aporta variabilidad ni utilidad

Traffic Emissions Unit:siempre "t.km-1.year-1", igual que la anterior (metadato técnico que no se utilizará)

Industrial Emissions Unit: siempre "t.year-1", unidad redundante, no se usará en el análisis

Detection Limit Unit: siempre "ug.m-3" o "ng.m-3", describe la unidad del límite de detección del equipo
    no es relevante para seleccionar estaciones ni para unir con mediciones

Duration Unit: siempre "day" o "hour", describe la unidad de duración del muestreo, pero no se usa en el análisis

Cadence Unit: siempre "day" o "hour", es un metadato técnico del inventario, no del contaminante

'''

'\nno son columnas vacías, pero presentan metadatos extremadamente técnicos sobre unidades de medida,\nformatos internos o descripciones redundantes\n\nno aportan información útil para el análisis de contaminación ni para la selección de estaciones y solo\nañaden ruido y complejidad al dataframe\n\ntras observar bien el data, voy a eliminar:\n\nAltitude Unit: siempre es "m", la unidad es redundante porque Altitude ya está en metros\n    no aporta información adicional\n\nInlet Height Unit: siempre es "m", describe la unidad de la altura de entrada del muestreador\n    no es relevante para el análisis de contaminación a nivel nacional\n\nBuilding Distance Unit: siempre es "m", la unidad es redundante, la distancia ya está en metros\n    no aporta variabilidad\n\nKerb Distance Unit: igual que la anterior: siempre "m"\n    no aporta info útil\n\nDistance Source Unit: siempre "m", es un metadato técnico del inventario, no del contaminante ni de la estación\n\nHeating Emissions Unit: siempr

In [63]:
cols_technical = [
    "Altitude Unit",
    "Inlet Height Unit",
    "Building Distance Unit",
    "Kerb Distance Unit",
    "Distance Source Unit",
    "Heating Emissions Unit",
    "Traffic Emissions Unit",
    "Industrial Emissions Unit",
    "Detection Limit Unit",
    "Duration Unit",
    "Cadence Unit"
]

df_stations = df_stations.drop(columns=[col for col in cols_technical if col in df_stations.columns])

print("Shape tras eliminar metadatos técnicos:", df_stations.shape)

Shape tras eliminar metadatos técnicos: (8564, 47)


In [64]:
df_stations.columns

Index(['Year', 'Air Quality Network', 'Air Quality Network Name', 'Timezone',
       'Air Quality Station EoI Code', 'Air Quality Station Nat Code',
       'Air Quality Station Name', 'Sampling Point Id', 'Air Pollutant',
       'Longitude', 'Latitude', 'Altitude', 'Air Quality Station Area',
       'Air Quality Station Type', 'Operational Activity Begin', 'Sample Id',
       'Inlet Height', 'Building Distance', 'Kerb Distance', 'Distance Source',
       'Main Emission Sources', 'Heating Emissions', 'Mobile',
       'Traffic Emissions', 'Industrial Emissions', 'Municipality',
       'Dispersion Local', 'Dispersion Regional', 'Distance Junction',
       'Heavy Duty Fraction', 'Height Facades', 'Street Width',
       'Traffic Speed', 'Traffic Volume', 'Process Id',
       'Process Activity Begin', 'Measurement Type', 'Measurement Method',
       'Measurement Equipment', 'Sampling Method', 'Analytical Technique',
       'Equivalence Demonstrated', 'Demonstration Report', 'Detection Limit'

In [65]:
# pese a una primera limpia y eliminar columnas vacías, constantes y metadatos técnicos, sigo teniendo muchas columnas (47)
# lo que voy a hacer es mirar una por una y ver las que realmente necesito para mi EDA

'''
necesito columnas que me aporten info para: identificar estaciones, localizar estaciones en el mapa, filtrar por tipo de estación (tráfico,
urbano, fondo…), seleccionar contaminantes relevantes (PM2.5, NO2, O3) y para unir este dataset con los datos diarios de contaminación

voy a conservar:

Year: año de referencia del inventario de estaciones, no es un rango temporal de mediciones, pero sirve para asegurar que las estaciones
    estaban operativas en el periodo de estudio

Air Pollutant: contaminante medido por esa estación (PM10, PM2.5, NO2, O3, metales…), es esencial para filtrar solo los contaminantes
    relevantes del proyecto

Air Quality Station EoI Code: código europeo único de la estación (EoI = “European Observation Identifier”), es la clave principal para
    unir estaciones con mediciones diarias

Air Quality Station Nat Code: código nacional de la estación, úil como identificador alternativo si el EoI no aparece en el dataset

Air Quality Station Name: nombre de la estación (“EL PICARRAL”, “PLAZA CASTILLA”…), facilita la interpretación y permite validaciones manuales

Sampling Point Id: identificador del punto de muestreo dentro de la estación, es importante porque una misma estación puede tener varios puntos
    para distintos contaminantes

Longitude y Latitude: ambas columnas muestran las coordenadas geográficas de la estación, son neecsarias para mapas, análisis espacial y 
    para comprobar si una estación es urbana, suburbana o rural

Municipality: municipio donde se ubica la estación, es útil para análisis descriptivos y para validar la localización

Air Quality Station Area: es la clasificación del entorno donde se encuentra la estación (urban, suburban, rural), es importante para
    para filtrar estaciones representativas del aire que respira la población

Air Quality Station Type: tipo de estación que es (traffic, background, industrial…), es fundamental para el EDA porque las estaciones de tráfico
    captan contaminación local, las estaciones de fondo urbano captan contaminación representativa de la ciudad, etc

Altitude: altitud de la estación, puede ser útil para análisis exploratorios o mapas

Main Emission Sources: fuente principal de emisiones en la zona (Transport, Industry, Heating…), ayuda a contextualizar por qué una estación
    mide ciertos niveles.

'''

'\nnecesito columnas que me aporten info para: identificar estaciones, localizar estaciones en el mapa, filtrar por tipo de estación (tráfico,\nurbano, fondo…), seleccionar contaminantes relevantes (PM2.5, NO2, O3) y para unir este dataset con los datos diarios de contaminación\n\nvoy a conservar:\n\nYear: año de referencia del inventario de estaciones, no es un rango temporal de mediciones, pero sirve para asegurar que las estaciones\n    estaban operativas en el periodo de estudio\n\nAir Pollutant: contaminante medido por esa estación (PM10, PM2.5, NO2, O3, metales…), es esencial para filtrar solo los contaminantes\n    relevantes del proyecto\n\nAir Quality Station EoI Code: código europeo único de la estación (EoI = “European Observation Identifier”), es la clave principal para\n    unir estaciones con mediciones diarias\n\nAir Quality Station Nat Code: código nacional de la estación, úil como identificador alternativo si el EoI no aparece en el dataset\n\nAir Quality Station Name:

In [66]:
# ahora voy a llevarlo a cabo, mantener las columnas que dije antes

cols_keep = [
    "Year",
    "Air Pollutant",
    "Air Quality Station EoI Code",
    "Air Quality Station Nat Code",
    "Air Quality Station Name",
    "Sampling Point Id",
    "Longitude",
    "Latitude",
    "Municipality",
    "Air Quality Station Area",
    "Air Quality Station Type",
    "Altitude",
    "Main Emission Sources"
]

df_stations = df_stations[cols_keep].copy()

print("Shape final estaciones:", df_stations.shape)


Shape final estaciones: (8564, 13)


In [67]:
df_stations.head()

,Year,Air Pollutant,Air Quality Station EoI Code,Air Quality Station Nat Code,Air Quality Station Name,Sampling Point Id,Longitude,Latitude,Municipality,Air Quality Station Area,Air Quality Station Type,Altitude,Main Emission Sources
0,2025,PM10,ES1044A,50297026,EL PICARRAL,SP_50297026_10_M,-0.87111,41.67028,ZARAGOZA,suburban,traffic,195.0,Transport
1,2025,NOX as NO2,ES1044A,50297026,EL PICARRAL,SP_50297026_12_8,-0.87111,41.67028,ZARAGOZA,suburban,traffic,195.0,Transport
2,2025,O3,ES1044A,50297026,EL PICARRAL,SP_50297026_14_6,-0.87111,41.67028,ZARAGOZA,suburban,traffic,195.0,Transport
3,2025,As in PM10,ES1044A,50297026,EL PICARRAL,SP_50297026_17_M,-0.87111,41.67028,ZARAGOZA,suburban,traffic,195.0,Transport
4,2025,Pb in PM10,ES1044A,50297026,EL PICARRAL,SP_50297026_19_M,-0.87111,41.67028,ZARAGOZA,suburban,traffic,195.0,Transport


In [68]:
df_stations.dtypes

Year                              int64
Air Pollutant                       str
Air Quality Station EoI Code        str
Air Quality Station Nat Code      int64
Air Quality Station Name            str
Sampling Point Id                   str
Longitude                       float64
Latitude                        float64
Municipality                        str
Air Quality Station Area            str
Air Quality Station Type            str
Altitude                        float64
Main Emission Sources               str
dtype: object

In [69]:
# he mirado los tipos de los datos y he visto que Air Quality Station Nat Code debe ser str, no int64
# Air Quality Station Nat Code es un código, no un número, puede tener ceros a la izquierda y los perderías si lo dejo como int
# los códigos numéricos de estaciones no se suman, no se restan, no se ordenan como números, por eso no tiene sentido que sean int


# vamos a convertirlo

df_stations["Air Quality Station Nat Code"] = df_stations["Air Quality Station Nat Code"].astype(str)

In [70]:
df_stations.dtypes

Year                              int64
Air Pollutant                       str
Air Quality Station EoI Code        str
Air Quality Station Nat Code        str
Air Quality Station Name            str
Sampling Point Id                   str
Longitude                       float64
Latitude                        float64
Municipality                        str
Air Quality Station Area            str
Air Quality Station Type            str
Altitude                        float64
Main Emission Sources               str
dtype: object

In [73]:
# en df_stations no hace falta renombrar ningun columna porque porque los nombres ya son claros y útiles

'''
por si no queda claro y para el que interese, hago traducción al español:

Index(['Year', 'Air Pollutant', 'Air Quality Station EoI Code',
       'Air Quality Station Nat Code', 'Air Quality Station Name',
       'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality',
       'Air Quality Station Area', 'Air Quality Station Type', 'Altitude',
       'Main Emission Sources'],
Index(['Año', 'Contaminante','Código EoI de la estación de calidad del aire',
    'Código nacional de la estación de calidad del aire', 'Nombre de la estación de calidad del aire',
    'ID del punto de muestreo', 'Longitud', 'Latitud', 'Municipio',
    'Área de la estación de calidad del aire', 'Tipo de estación de calidad del aire','Altitud',
    'Fuentes principales de emisión'])    
'''

"\npor si no queda claro y para el que interese, hago traducción al español:\n\nIndex(['Year', 'Air Pollutant', 'Air Quality Station EoI Code',\n       'Air Quality Station Nat Code', 'Air Quality Station Name',\n       'Sampling Point Id', 'Longitude', 'Latitude', 'Municipality',\n       'Air Quality Station Area', 'Air Quality Station Type', 'Altitude',\n       'Main Emission Sources'],\nIndex(['Año', 'Contaminante','Código EoI de la estación de calidad del aire',\n    'Código nacional de la estación de calidad del aire', 'Nombre de la estación de calidad del aire',\n    'ID del punto de muestreo', 'Longitud', 'Latitud', 'Municipio',\n    'Área de la estación de calidad del aire', 'Tipo de estación de calidad del aire','Altitud',\n    'Fuentes principales de emisión'])    \n"

In [71]:
'''
# voy a filtrar por año (2011-2019)
# la columna "Year" indica el año del inventario de estaciones, no el año de la medición
# aún así, necesitamos asegurarnos de que la estación estaba activa durante el periodo de estudio

df_stations = df_stations[df_stations["Year"].between(2011, 2019)]


df_stations
'''

'''
lo de arriba está en comentario porque:

durante la limpieza del dataset de estaciones (df_stations) detecté que todos los registros
pertenecían únicamente al año 2025. Al investigar por qué no aparecían estaciones históricas
(2011-2019), confirmé que el dataflow oficial E1a había sido retirado por la Agencia Europea
de Medio Ambiente a finales de 2024, lo que inicialmente generó confusión porque parecía
imposible continuar el análisis espacial sin esos metadatos.
'''

# debería haber mirado desde el principio qué años contenía la columna Year, pero no fue así,
# a partir de ahora lo comprobaré siempre

'''
sin embargo, al revisar la estructura de los identificadores, observé que los SamplingPoint
de los contaminantes contienen un código numérico estable de 8 dígitos que coincide con el
Air Quality Station Nat Code y con el núcleo de Sampling Point Id del dataset de estaciones
2025. Gracias a esta correspondencia, es posible identificar y emparejar las estaciones
históricas con sus coordenadas y metadatos, permitiendo continuar el EDA con normalidad
'''

'\nsin embargo, al revisar la estructura de los identificadores, observé que los SamplingPoint\nde los contaminantes contienen un código numérico estable de 8 dígitos que coincide con el\nAir Quality Station Nat Code y con el núcleo de Sampling Point Id del dataset de estaciones\n2025. Gracias a esta correspondencia, es posible identificar y emparejar las estaciones\nhistóricas con sus coordenadas y metadatos, permitiendo continuar el EDA con normalidad\n'

In [ ]:
# GUARDO LOS DATASETS LIMPIOS PARA NOTEBOOK 04

os.makedirs("../data/context_clean", exist_ok=True)

df_hospi.to_csv("../data/context_clean/hospitalizations_clean.csv", index=False)
df_mortality.to_csv("../data/context_clean/mortality_clean.csv", index=False)
df_population.to_csv("../data/context_clean/population_clean.csv", index=False)
df_stations.to_csv("../data/context_clean/stations_clean.csv", index=False)

print("Datasets limpios guardados correctamente.")

Datasets limpios guardados correctamente.


In [75]:
df_mortality

,region,year,deaths
14,ES,2011,42125
15,ES,2012,47237
16,ES,2013,42444
17,ES,2014,43699
18,ES,2015,51711
19,ES,2016,46649
20,ES,2017,51413
21,ES,2018,53476
22,ES,2019,47470


In [ ]:
# CIERRE DEL NOTEBOOK 03

'''
en este notebook hemos realizado el exploratorio inicial y la limpieza completa de los
datasets de contexto: hospitalizaciones, mortalidad, población y estaciones de calidad
del aire. Cada uno ha quedado reducido a las columnas esenciales, con tipos correctos,
sin duplicados ni metadatos irrelevantes.

en el caso del dataset de estaciones (df_stations), detecté que todos los registros
pertenecían al inventario de 2025 debido a la retirada del dataflow histórico E1a por
parte de la Agencia Europea de Medio Ambiente
sin embargo, confirmé que los códigos nacionales de estación (NatCode) son estables en el tiempo
y permiten identificar las estaciones históricas presentes en los datasets de contaminantes

finalizamos aquí el Notebook 03: ya hemos dejado preparados y limpios todos los datasets de contexto
el siguiente paso (extraer el NatCode de los contaminantes, unirlos con las estaciones y comenzar el
análisis integrado) se realizará en el Notebook 04.
'''
